> **Note:** Before you continue reading, it is important to note that a more detailed analysis and explanation (including EDA and recommendations) will be provided in the *ab_testing_analysis* PDF file. This notebook is kept concise for readability.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import (proportion_effectsize, proportions_ztest, confint_proportions_2indep)
import numpy as np
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.anova import anova_lm
import statsmodels.formula.api as smf

___
### 2. Read & Analyze Data

### Data Dictionary

**Note:** PSA stands for **Public Service Announcement**.

| Column | Description |
|----------|-------------|
| **Index** | Row index |
| **user id** | Unique user identifier |
| **test group** | If `"ad"`, the person saw the advertisement; if `"psa"`, they only saw the public service announcement |
| **converted** | `True` if the person bought the product; otherwise `False` |
| **total ads** | Total number of advertisements seen by the person |
| **most ads day** | Day on which the person saw the highest number of advertisements |
| **most ads hour** | Hour of the day during which the person saw the highest number of advertisements |

In [ ]:
df = pd.read_csv("marketing_AB.csv")

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df = df.drop("Unnamed: 0", axis=1)
df

In [ ]:
df.describe().T

In [ ]:
df.info()

___
### 3. Data Cleaning

In [ ]:
df = df.rename(columns={
    'user id': 'user_id',
    'test group': 'test_group',
    'total ads': 'total_ads',
    'most ads day': 'most_ads_day',
    'most ads hour': 'most_ads_hour'
})

df['test_group'] = df['test_group'].astype('category')
df["converted"] = df["converted"].astype(int)
df['most_ads_day'] = df['most_ads_day'].astype("category")
df['most_ads_hour'] = df['most_ads_hour'].astype("category")

In [ ]:
df.info()

___
### 4. Basic EDA

In [ ]:
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
print("Test group:", df['test_group'].unique())
print("Converted:", df['converted'].unique())
print("Most ads day:", df['most_ads_day'].unique())

In [ ]:
df

In [ ]:
df["test_group"].value_counts()

In [ ]:
df["test_group"].value_counts(normalize=True) 

In [ ]:
df.groupby("test_group")["converted"].sum()

In [ ]:
# Data
group_counts = df["test_group"].value_counts()
group_conversion = df.groupby("test_group", observed=False)["converted"].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# -----------------------
# Left: Distribution of users
# -----------------------
sns.barplot(
    x=group_counts.index,
    y=group_counts.values,
    ax=axes[0],
    palette="Blues",
    hue=group_counts.index,
    legend=False
)

axes[0].set_title("Distribution of Test Groups")
axes[0].set_xlabel("Test Group")
axes[0].set_ylabel("Number of Users")

# -----------------------
# Right: Conversion rate
# -----------------------
sns.barplot(
    x=group_conversion.index,
    y=group_conversion.values,
    ax=axes[1],
    palette="Reds",
    hue=group_conversion.index,
    legend=False
)

axes[1].set_title("Conversion Rate by Test Group")
axes[1].set_xlabel("Test Group")
axes[1].set_ylabel("Conversion Rate")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Reorder data
day_counts = df["most_ads_day"].value_counts().reindex(weekday_order)
day_conversions = df.groupby("most_ads_day", observed=False)["converted"].sum().reindex(weekday_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. User distribution
sns.barplot(
    x=day_counts.index,
    y=day_counts.values,
    hue=day_counts.index,
    palette="Blues",
    ax=axes[0],
    legend=False
)

axes[0].set_title("User Distribution by Most Ads Day")
axes[0].set_ylabel("Number of Users")
axes[0].set_xlabel("Day")

# 2. Conversions
sns.barplot(
    x=day_conversions.index,
    y=day_conversions.values,
    hue=day_conversions.index,
    palette="Reds",
    ax=axes[1],
    legend=False
)

axes[1].set_title("Conversions by Most Ads Day")
axes[1].set_ylabel("Number of Conversions")
axes[1].set_xlabel("Day")

plt.tight_layout()
plt.show()

In [ ]:
cr_day_sample = (
    df.groupby([df['most_ads_day'], df['test_group']], observed=False)['converted']
      .mean()
      .unstack()
      .reindex(weekday_order))

cr_day_sample.plot(kind='bar')
plt.Figure(figsize=(10, 3))
plt.xlabel('Weekday')
plt.ylabel('Conversion rate')
plt.show()

In [ ]:
# Data
hour_counts = df["most_ads_hour"].value_counts().sort_index()
hour_conversions = df.groupby("most_ads_hour", observed=False)["converted"].sum()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# -----------------------
# Left: User distribution
# -----------------------
sns.barplot(
    x=hour_counts.index,
    y=hour_counts.values,
    ax=axes[0],
    palette="Blues",
    hue=hour_counts.index,
    legend=False
)

axes[0].set_title("User Distribution by Most Ads Hour")
axes[0].set_xlabel("Hour")
axes[0].set_ylabel("Number of Users")

# -----------------------
# Right: Conversions
# -----------------------
sns.barplot(
    x=hour_conversions.index,
    y=hour_conversions.values,
    ax=axes[1],
    palette="coolwarm",
    hue=hour_conversions.index,
    legend=False
)

axes[1].set_title("Conversions by Most Ads Hour")
axes[1].set_xlabel("Hour")
axes[1].set_ylabel("Number of Conversions")

plt.tight_layout()
plt.show()

In [ ]:
# Filter out top 1% extreme values
percentile_99 = df['total_ads'].quantile(0.99)
df_clean = df[df['total_ads'] <= percentile_99].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# -----------------------
# BEFORE outlier removal
# -----------------------
sns.boxplot(
    x='total_ads',
    y='test_group',
    data=df,
    palette='pastel',
    hue='test_group',
    legend=False,
    ax=axes[0]
)

axes[0].set_title('Total Ads per User (BEFORE Outliers)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Total Ads')
axes[0].set_ylabel('Test Group')

# -----------------------
# AFTER outlier removal
# -----------------------
sns.boxplot(
    x='total_ads',
    y='test_group',
    data=df_clean,
    palette='Set2',
    hue='test_group',
    legend=False,
    ax=axes[1]
)

axes[1].set_title('Total Ads per User (AFTER Outliers)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Total Ads')
axes[1].set_ylabel('Test Group')

plt.tight_layout()
plt.show()

# Distribution of Histogram of Total Ads Seen per User
plt.figure(figsize=(10, 4))
sns.histplot(data=df_clean, x='total_ads', hue='test_group', bins=30, kde=True, palette='Set2')
plt.title('Distribution of Total Ads Seen per User', fontsize=14, fontweight='bold')
plt.xlabel( 'Total Ads')
plt.show()

___
### 5. Hypothesis Testing

#### 5.1 To evaluate whether the advertisement truly increases the conversion rate.

**Null Hypothesis (H0):** The conversion rate in the ad group is less than or equal to that of the PSA group

**Alternative Hypothesis (H1):** The conversion rate in the ad group is higher than that of the PSA group

We will use a two-sample z-test for proportions to test this hypothesis.


In [ ]:
summary = df.groupby("test_group", observed=False)["converted"].agg(["sum", "count"])

converted_ad = summary.loc["ad", "sum"]
converted_psa = summary.loc["psa", "sum"]

n_ad = summary.loc["ad", "count"]
n_psa = summary.loc["psa", "count"]

z, p = proportions_ztest(
    [converted_ad, converted_psa],
    [n_ad, n_psa],
    alternative="larger"
)

ci_low, ci_upp = confint_proportions_2indep(
    converted_ad, n_ad,
    converted_psa, n_psa,
    method="wald"
)

lift_pp = (converted_ad/n_ad - converted_psa/n_psa) * 100
result_df = pd.DataFrame({
    'z_stat': [z],
    'p_value': [p],
    'ci_lower_pp': [ci_low * 100],
    'ci_upper_pp': [ci_upp * 100],
    'lift_pp': [lift_pp]
})

result_df

**Bootstrap analysis to assess the robustness of the results**

In [ ]:
# ad = df[df['test_group'] == 'ad']['converted'].values
# psa = df[df['test_group'] == 'psa']['converted'].values

# n_boot = 1000
# boot_diffs = []

# for _ in range(n_boot):
#     ad_sample = np.random.choice(ad, size=len(ad), replace=True)
#     psa_sample = np.random.choice(psa, size=len(psa), replace=True)
    
#     boot_diffs.append(ad_sample.mean() - psa_sample.mean())

# boot_diffs = np.array(boot_diffs)

# ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])
# p_value = np.mean(boot_diffs <= 0)

# bootstrap_primary = pd.DataFrame({
#     'p_value': [f"{p_value:.3f}"],
#     'ci_lower_pp': [ci_low * 100],
#     'ci_upper_pp': [ci_high * 100],
#     'lift_pp': [(ad.mean() - psa.mean()) * 100]
# })

# bootstrap_primary

**Conclusions**
- Since the p-value is much smaller than 0.05, we reject the null hypothesis. This indicates that there is a statistically significant difference in conversion rates between the ad group and the PSA group. In other words, showing advertisements has a significant effect on increasing conversions in this experiment.

- A bootstrap resampling analysis further confirms this finding. The estimated uplift remains consistently positive across resamples, with a confidence interval that does not include zero, indicating that the result is robust and not driven by sampling variability.

#### 5.2 To evaluate whether the experiment is balanced, meaning whether users were randomly and evenly distributed between the ad and PSA groups across different days, and whether the conversion effect varies across days.

**Null Hypothesis (H0):** The test group and most_ads_day are independent (there is no association between group assignment and day distribution).

**Alternative Hypothesis (H1):** The test group and most_ads_day are not independent (there is an association between group assignment and day distribution).

We will use a Chi-square test of independence to test this hypothesis.

In [ ]:
pd.crosstab(df['most_ads_day'], df['test_group'])

In [ ]:
table = pd.crosstab(df['most_ads_day'], df['test_group'])

chi2, p_value, _, _ = chi2_contingency(table)

print(f'P-value: {p_value}')

In [ ]:
pvals, ci_lower, ci_upper, lifts = [], [], [], []

for day in weekday_order:
    sub = df[df['most_ads_day'] == day]
    
    summary = sub.groupby('test_group')['converted'].agg(['sum', 'count'])
    
    counts = summary['sum']
    nobs = summary['count']
    
    # Two-sample z-test (ad vs psa) for this day
    pvals.append(proportions_ztest(counts, nobs)[1])
    
    # Confidence interval for difference in proportions
    low, high = confint_proportions_2indep(
        counts['ad'], nobs['ad'],
        counts['psa'], nobs['psa']
    )
    ci_lower.append(low)
    ci_upper.append(high)
    
    # Relative lift (%)
    lift = (counts['ad'] / nobs['ad'] - counts['psa'] / nobs['psa']) / \
           (counts['psa'] / nobs['psa']) * 100
    lifts.append(lift)

# Multiple testing correction (FDR - Benjamini/Hochberg)
reject, pvals_corr, _, _ = multipletests(pvals, method='fdr_bh')

# Final results table
daily_results = pd.DataFrame({
    'day': weekday_order,
    'p_value': pvals,
    'p_value_fdr': pvals_corr,
    'ci_lower': ci_lower,
    'ci_upper': ci_upper,
    'lift_percent': lifts,
    'significant': reject
})

# Extract significant days (after correction)
significant_days = daily_results.loc[daily_results['significant'], 'day']

daily_results

**Conclusions**
- Based on the Chi-square test of independence, the null hypothesis is rejected, indicating that there is an association between test group (ad vs PSA) and most_ads_day. This suggests that the distribution of users across days is not fully independent of group assignment.
- Post-hoc analysis using two-sample proportion z-tests with FDR correction shows that the advertisement effect varies across days of the week. Significant positive conversion lifts are observed on Monday, Tuesday, Wednesday, Friday, and Saturday, while no statistically significant differences are found on Thursday and Sunday.
- The confidence intervals for significant days are generally above zero, indicating a positive and measurable effect of the advertisement during those periods.

#### 5.3 To further evaluate the effect of advertisement on conversion and whether this effect differs across days using a two-way ANOVA model.

In [ ]:
model_anova = smf.ols('converted ~ C(test_group) * C(most_ads_day)', data=df).fit()

anova_results = anova_lm(model_anova)

anova_results

**Conclusions**
- The ANOVA results further confirm these findings. There are significant main effects of both test group and day of week on conversion rates, indicating that the advertisement influences conversion overall and that conversion behavior differs across days. Importantly, the significant interaction effect between test group and day of week suggests that the effectiveness of the advertisement varies depending on the day.

#### 5.4 To evaluate whether the hour of advertisement exposure affects conversion probability

In [ ]:
data = df[(df['test_group'] == 'ad') & (df['most_ads_day'].isin(significant_days))].copy()

data['converted'] = data['converted'].astype(int)

model = smf.logit('converted ~ C(most_ads_hour)', data=data).fit(disp=False)

print(f'LLR p-value: {model.llr_pvalue}')

In [ ]:
hours = pd.DataFrame({'most_ads_hour': range(24)})

pred = model.get_prediction(hours).summary_frame()
pred = pred.rename(columns={'predicted': 'predicted_cr'})

hourly_effect = pd.concat([hours, pred], axis=1)

hourly_effect_sorted = hourly_effect.sort_values('predicted_cr', ascending=False)

hourly_effect_sorted

**Conclusion**
- Logistic regression indicates a statistically significant relationship between conversion probability and hour of day within the ad group on days where CR(ad) > CR(psa). This suggests that ad effectiveness varies meaningfully across different hours.

- Predicted conversion probabilities are highest in the afternoon and early evening, with a consistent peak between 14:00–17:00 and a secondary high around 20:00–21:00. The lowest conversion probabilities occur during late night and early morning hours (approximately 01:00–04:00). Narrow confidence intervals across most hours indicate stable estimates, supporting reliable ranking of hourly performance.

 

___
### 6. Recommendations
I will include this section in the ab_testing_analysis PDF file, as I am keeping the text here concise to avoid clutter